# Kafka issue types: BERT tuning

In [ ]:
!pip install transformers datasets accelerate jira tqdm scikit-learn pandas numpy torch

In [2]:
import re
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from tqdm.auto import tqdm
from jira import JIRA
import json
from datetime import datetime

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [3]:
jira = JIRA('https://issues.apache.org/jira')
jql = 'project=KAFKA'

all_issues = []
start = 0
batch_size = 100

while True:
    issues = jira.search_issues(
        jql,
        startAt=start,
        maxResults=batch_size,
        fields=['summary', 'issuetype', 'status', 'created', 'reporter', 'description'])
    if not issues:
        break
    all_issues.extend(issues)
    print(f"Found {len(all_issues)} issues")
    start += batch_size
print(f"Found {len(all_issues)} issues total")

def extract_issue(issue):
    return {
        'bug': issue.key,
        'summary': issue.fields.summary or '',
        'description': issue.fields.description or '',
        'type': issue.fields.issuetype.name,
        'status': issue.fields.status.name,
        'created': issue.fields.created,
        'reporter': issue.fields.reporter.displayName if issue.fields.reporter else 'Unknown'
    }

data = [extract_issue(issue) for issue in all_issues]
df = pd.DataFrame(data)
df.to_csv('Kafka_issues.csv', index=False)

Found 19516 issues total


In [4]:
df['text'] = (df['summary'].fillna('') + ' ' + df['description'].fillna('')).str.strip()
df = df[df['text'].str.len() > 0].reset_index(drop=True)

MIN_CLASS_COUNT = 50
type_counts = df['type'].value_counts()
rare_types = type_counts[type_counts < MIN_CLASS_COUNT].index
if len(rare_types) > 0:
    print(f"Collapsing rare issue types into 'Other': {list(rare_types)}")
    df['type'] = df['type'].apply(lambda t: 'Other' if t in rare_types else t)

categories = sorted(df['type'].unique().tolist())
label_to_idx = {label: i for i, label in enumerate(categories)}
df['label'] = df['type'].map(label_to_idx)

print(f"\nModeling categories: {categories}")
print(df['type'].value_counts())


Modeling categories: ['Bug', 'Improvement', 'New Feature', 'Sub-task', 'Task', 'Test', 'Wish']
type
Bug            8905
Improvement    5021
Sub-task       3325
Task            872
Test            651
New Feature     607
Wish            135
Name: count, dtype: int64


In [5]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

In [6]:
class BertJiraDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [7]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_dataset = BertJiraDataset(train_texts, train_labels, tokenizer)
test_dataset = BertJiraDataset(test_texts, test_labels, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [8]:
learning_rates = [1e-5, 2e-5, 3e-5]
epochs_list = [3, 4]

all_results = []
best_macro_f1 = 0
best_model = None
best_params = None

for lr in learning_rates:
    for epochs in epochs_list:
        print(f"\nLR={lr}, Epochs={epochs}")

        model = BertForSequenceClassification.from_pretrained(
            'bert-base-uncased',
            num_labels=len(categories)
        )
        model = model.to(device)

        optimizer = AdamW(model.parameters(), lr=lr)
        total_steps = len(train_loader) * epochs
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=0,
            num_training_steps=total_steps
        )

        for epoch in range(epochs):
            model.train()
            total_loss = 0
            progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

            for batch in progress:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                optimizer.zero_grad()

                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )

                loss = outputs.loss
                total_loss += loss.item()

                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                scheduler.step()

                progress.set_postfix(loss=loss.item())

            avg_loss = total_loss / len(train_loader)
            print(f"Epoch {epoch+1} average loss: {avg_loss:.4f}")

        model.eval()
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for batch in tqdm(test_loader, desc="Evaluating"):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                )

                logits = outputs.logits
                preds = torch.argmax(logits, dim=1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        accuracy = np.mean(np.array(all_preds) == np.array(all_labels))
        macro_f1 = f1_score(all_labels, all_preds, average='macro')
        weighted_f1 = f1_score(all_labels, all_preds, average='weighted')

        result = {
            'learning_rate': lr,
            'epochs': epochs,
            'batch_size': 16,
            'max_len': 128,
            'accuracy': float(accuracy),
            'macro_f1': float(macro_f1),
            'weighted_f1': float(weighted_f1)
        }
        all_results.append(result)

        print(f"Accuracy: {accuracy:.4f}")
        print(f"Macro F1: {macro_f1:.4f}")
        print(f"Weighted F1: {weighted_f1:.4f}")

        if macro_f1 > best_macro_f1:
            best_macro_f1 = macro_f1
            best_model = model
            best_params = {'lr': lr, 'epochs': epochs}
            model.save_pretrained('./best_model')
            tokenizer.save_pretrained('./best_model')


LR=1e-05, Epochs=3


Epoch 1 average loss: 1.0848


Epoch 2 average loss: 0.9050


Epoch 3 average loss: 0.8120


Accuracy: 0.6811
Macro F1: 0.3835
Weighted F1: 0.6596



LR=1e-05, Epochs=4


... (22 outputs left out)

Epoch 2 average loss: 0.8016


Epoch 3 average loss: 0.5832


Epoch 4 average loss: 0.3963


Accuracy: 0.6790
Macro F1: 0.4515
Weighted F1: 0.6735


In [9]:
results_df = pd.DataFrame(all_results)
print(results_df.to_string(index=False))

best_idx = results_df['macro_f1'].idxmax()
print("\nBest model:")
print(f"Learning rate: {results_df.iloc[best_idx]['learning_rate']}")
print(f"Epochs: {int(results_df.iloc[best_idx]['epochs'])}")
print(f"Macro F1: {results_df.iloc[best_idx]['macro_f1']:.4f}")
print(f"Accuracy: {results_df.iloc[best_idx]['accuracy']:.4f}")

 learning_rate  epochs  batch_size  max_len  accuracy  macro_f1  weighted_f1
       0.00001       3          16      128  0.681096  0.383522     0.659607
       0.00001       4          16      128  0.687244  0.418136     0.672533
       0.00002       3          16      128  0.682377  0.418168     0.668923
       0.00002       4          16      128  0.686988  0.443953     0.678040
       0.00003       3          16      128  0.688525  0.439253     0.679054
       0.00003       4          16      128  0.679047  0.451451     0.673508

Best model:
Learning rate: 3e-05
Epochs: 4
Macro F1: 0.4515
Accuracy: 0.6790


In [10]:
results_df.to_csv('parameter_comparison.csv', index=False)

with open('all_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print(f"Best parameters: LR={best_params['lr']}, Epochs={best_params['epochs']}")

Best parameters: LR=3e-05, Epochs=4
